# Crawling Data

Crawling data dilakukan untuk mengumpulkan URL berita yang akan digunakan pada proses scraping. Pada tahap ini, sumber data yang digunakan adalah RSS (Really Simple Syndication) dari Detik.com, yaitu RSS kategori Sport dan Finance.

RSS digunakan untuk memperoleh daftar berita terbaru beserta URL masing-masing berita. URL yang diperoleh kemudian digunakan sebagai input pada tahap scraping untuk mengambil isi berita.

Dalam proses pengumpulan data, masing-masing RSS dikumpulkan sebanyak 100 URL berita, sehingga diperoleh 200 URL yang terdiri dari 100 berita kategori Sport dan 100 berita kategori Finance.

## Pengambilan URL Berita

Library feedparser digunakan untuk membaca dan mengambil data dari RSS. RSS Sport dan Finance Detik digunakan sebagai sumber untuk mendapatkan URL berita.

Setiap data berita pada RSS memiliki URL yang dapat diambil melalui atribut entry.link. URL tersebut kemudian disimpan ke dalam dua list, yaitu link_sport dan link_finance.

In [2]:
import feedparser

feed_sport = feedparser.parse("https://sport.detik.com/rss")
feed_finance = feedparser.parse("https://finance.detik.com/rss")

link_sport = [entry.link for entry in feed_sport.entries]
link_finance = [entry.link for entry in feed_finance.entries]

print("Sport:", len(link_sport))
print("Finance:", len(link_finance))

Sport: 100
Finance: 100


## Hasil Crawling

Hasil crawling menunjukkan bahwa sistem berhasil memperoleh 100 URL berita dari kategori Sport dan 100 URL berita dari kategori Finance. Jadi terdapat 200 URL berita yang digunakan sebagai sumber pada tahap scraping.

Data yang diperoleh pada tahap crawling belum berupa isi berita, tetapi masih berupa URL yang mengarah ke halaman artikel. URL tersebut selanjutnya diproses pada tahap scraping untuk mengambil teks berita.

# Scraping Isi Berita

URL yang diperoleh dari proses crawling selanjutnya digunakan untuk mengambil isi dari setiap halaman berita. Pada tahap ini digunakan library trafilatura.

Fungsi fetch_url() digunakan untuk mengambil halaman dari URL berita, sedangkan extract() digunakan untuk mengambil teks utama dari halaman tersebut. Setiap berita yang berhasil diekstrak kemudian disimpan bersama label kategorinya, yaitu sport atau finance.

In [3]:
import trafilatura
import time
import random

def ekstrak_dataset(links, label):
    data = []
    gagal = []

    for i, link in enumerate(links, 1):
        downloaded = trafilatura.fetch_url(link)
        isi = None
        if downloaded:
            isi = trafilatura.extract(downloaded, favor_recall=True)

        if isi:
            data.append({"isi_berita": isi, "label": label})
        else:
            gagal.append(link)

        print(f"[{label}] {i}/{len(links)}", end="\r")
        time.sleep(random.uniform(2, 5))
    print()
    
    return data, gagal

data_sport, gagal_sport = ekstrak_dataset(link_sport, "sport")
data_finance, gagal_finance = ekstrak_dataset(link_finance, "finance")

print(f"Data Sport Berhasil di Ekstrak: {len(data_sport)}, Gagal: {len(gagal_sport)}")
print(f"Data Finance Berhasil di Ekstrak: {len(data_finance)}, Gagal: {len(gagal_finance)}")

[sport] 100/100
[finance] 100/100
Data Sport Berhasil di Ekstrak: 100, Gagal: 0
Data Finance Berhasil di Ekstrak: 100, Gagal: 0


Berdasarkan hasil scraping, seluruh 100 berita Sport dan 100 berita Finance berhasil diekstrak tanpa adanya data yang gagal. Maka dari itu, diperoleh 100 data berita yang dapat digunakan untuk tahap pengolahan selanjutnya.

# Pembentukan Dataset

In [4]:
import pandas as pd

data_sport = data_sport[:100]
data_finance = data_finance[:100]
df = pd.DataFrame(data_sport + data_finance)
df.insert(0, "id", range(1, len(df) + 1))

print(df.shape)
df.head()

(200, 3)


,id,isi_berita,label
0,1,Persib Bandung mengawali kiprahnya di AFC Cham...,sport
1,2,"Alwi Farhan, Moh Zaki Ubaidillah dan Muhamad Y...",sport
2,3,Tottenham Hotspur akhirnya kembali mencetak go...,sport
3,4,Yan Diomande tampil menawan saat Real Madrid m...,sport
4,5,"Pelatih Timnas Jerman, Juergen Klopp kabarnya ...",sport


In [5]:
df.to_csv("dataset_berita.csv", index=False, encoding="utf-8")
print("Selesai")

Selesai


Pada tahap ini, data berita dari kategori Sport dan Finance masing-masing diambil sebanyak 100 data. Kedua data tersebut kemudian digabungkan menjadi satu dataset menggunakan Pandas. Setiap data diberi nomor ID secara berurutan, sehingga diperoleh 100 data berita dalam format CSV yang siap digunakan.